# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import json
import os
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import AIMessage, BaseMessage, ToolMessage
from lib.parsers import PydanticOutputParser
from lib.state_machine import Run
from lib.tooling import tool

In [3]:
for env_path in [Path("../../../.env"), Path("../../.env"), Path(".env")]:
    if env_path.exists():
        load_dotenv(env_path)
        break
else:
    load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY", OPENAI_API_KEY)
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL") or os.getenv(
    "OPENAI_API_BASE", "https://openai.vocareum.com/v1"
)
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["CHROMA_OPENAI_API_KEY"] = CHROMA_OPENAI_API_KEY
os.environ["OPENAI_BASE_URL"] = OPENAI_BASE_URL
os.environ["OPENAI_API_BASE"] = OPENAI_BASE_URL

In [4]:
COLLECTION_NAME = "udaplay"
CHROMA_PATH = "chromadb"

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    api_base=OPENAI_BASE_URL,
)
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

judge_llm = LLM(model="gpt-4o-mini", temperature=0.0, api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Explanation of the evaluation result")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
@tool
def retrieve_game(query: str) -> List[Dict[str, Any]]:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(query_texts=[query], n_results=3, include=["metadatas", "distances"])

    retrieved = []
    for metadata, distance in zip(results["metadatas"][0], results["distances"][0]):
        retrieved.append({
            **metadata,
            "similarity": round(1 - distance, 3),
        })

    return retrieved

#### Evaluate Retrieval Tool

In [6]:
@tool
def evaluate_retrieval(question: str, retrieved_docs: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Question: {question}\n\n"
        f"Retrieved documents:\n{json.dumps(retrieved_docs, indent=2)}"
    )

    response = judge_llm.invoke(prompt, response_format=EvaluationReport)
    parser = PydanticOutputParser(model_class=EvaluationReport)
    report = parser.parse(response)

    return report.model_dump()

#### Game Web Search Tool

In [7]:
@tool
def game_web_search(question: str) -> Dict[str, Any]:
    """
    Search the web for video game information when internal knowledge is insufficient.
    args:
    - question: a question about game industry.
    """
    search_result = tavily_client.search(
        query=question,
        search_depth="advanced",
        include_answer=True,
        include_raw_content=False,
        include_images=False,
    )

    return {
        "answer": search_result.get("answer", ""),
        "results": [
            {
                "title": item.get("title"),
                "url": item.get("url"),
                "content": item.get("content"),
                "score": item.get("score"),
            }
            for item in search_result.get("results", [])
        ],
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": question,
        },
    }

### Agent

In [8]:
AGENT_INSTRUCTIONS = """
You are UdaPlay, an AI research agent for the video game industry.

Workflow for every user question:
1. Call retrieve_game with the user's question.
2. Call evaluate_retrieval with the question and retrieved documents.
3. If evaluation says documents are NOT useful, call game_web_search.
4. Produce a final answer using the best available source.

Rules:
- Prefer internal database results when they are sufficient.
- Cite sources clearly (game name/platform/year for internal docs, URLs for web results).
- Be concise but complete.
- If using web search, mention that internal knowledge was insufficient.
""".strip()

udaplay_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=AGENT_INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.2,
)

### Reporting System

In [9]:
def extract_tool_calls(messages: List[BaseMessage]) -> List[Dict[str, Any]]:
    tool_usage = []
    pending_calls = {}

    for message in messages:
        if isinstance(message, AIMessage) and message.tool_calls:
            for call in message.tool_calls:
                pending_calls[call.id] = {
                    "tool": call.function.name,
                    "arguments": json.loads(call.function.arguments),
                    "result": None,
                }
        elif isinstance(message, ToolMessage) and message.tool_call_id in pending_calls:
            try:
                pending_calls[message.tool_call_id]["result"] = json.loads(message.content)
            except json.JSONDecodeError:
                pending_calls[message.tool_call_id]["result"] = message.content
            tool_usage.append(pending_calls.pop(message.tool_call_id))

    return tool_usage


def extract_final_answer(messages: List[BaseMessage]) -> str:
    for message in reversed(messages):
        if isinstance(message, AIMessage) and message.content and not message.tool_calls:
            return message.content
    return "No final answer generated."


def print_agent_report(query: str, run: Run) -> None:
    final_state = run.get_final_state() or {}
    messages = final_state.get("messages", [])
    tool_usage = extract_tool_calls(messages)
    final_answer = extract_final_answer(messages)

    print("=" * 80)
    print(f"QUERY: {query}")
    print("=" * 80)

    print("\nTOOL USAGE:")
    if not tool_usage:
        print("- No tools were called.")
    for idx, usage in enumerate(tool_usage, start=1):
        print(f"\n{idx}. {usage['tool']}")
        print(f"   Arguments: {json.dumps(usage['arguments'], ensure_ascii=False)}")
        print(f"   Result: {json.dumps(usage['result'], ensure_ascii=False)[:500]}...")

    print("\nFINAL ANSWER:")
    print(final_answer)

    print("\nCITATIONS:")
    citations = []
    for usage in tool_usage:
        if usage["tool"] == "retrieve_game" and isinstance(usage["result"], list):
            for doc in usage["result"]:
                citations.append(
                    f"[Internal DB] {doc.get('Name')} ({doc.get('Platform')}, {doc.get('YearOfRelease')})"
                )
        elif usage["tool"] == "game_web_search" and isinstance(usage["result"], dict):
            for item in usage["result"].get("results", [])[:3]:
                citations.append(f"[Web] {item.get('title')} - {item.get('url')}")

    if citations:
        for citation in citations:
            print(f"- {citation}")
    else:
        print("- No explicit citations found.")

    print("\nRUN METADATA:")
    print(json.dumps(run.metadata, indent=2))
    print("=" * 80)

In [10]:
query_1 = "When Pokémon Gold and Silver was released?"
run_1 = udaplay_agent.invoke(query_1, session_id="udaplay-demo")
print_agent_report(query_1, run_1)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
QUERY: When Pokémon Gold and Silver was released?

TOOL USAGE:

1. retrieve_game
   Arguments: {"query": "Pokémon Gold and Silver release date"}
   Result: [{"Name": "Pokémon Gold and Silver", "YearOfRelease": 1999, "Description": "Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.", "Platform": "Game Boy Color", "Publisher": "Nintendo", "Genre": "Role-playing", "similarity": 0.866}, {"Description": "Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.", "YearOfRelease": 2002, "Name": "Pokémon Ruby and Sapphire", "Publisher": "Nintendo", "Genre": "Role-playing",

In [11]:
query_2 = "Which one was the first 3D platformer Mario game?"
run_2 = udaplay_agent.invoke(query_2, session_id="udaplay-demo")
print_agent_report(query_2, run_2)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
QUERY: Which one was the first 3D platformer Mario game?

TOOL USAGE:

1. retrieve_game
   Arguments: {"query": "Pokémon Gold and Silver release date"}
   Result: [{"Name": "Pokémon Gold and Silver", "YearOfRelease": 1999, "Description": "Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.", "Platform": "Game Boy Color", "Publisher": "Nintendo", "Genre": "Role-playing", "similarity": 0.866}, {"Description": "Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.", "YearOfRelease": 2002, "Name": "Pokémon Ruby and Sapphire", "Publisher": "Nintendo", "Genre": "Role-pl

In [12]:
query_3 = "Was Mortal Kombat X released for Playstation 5?"
run_3 = udaplay_agent.invoke(query_3, session_id="udaplay-demo")
print_agent_report(query_3, run_3)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
QUERY: Was Mortal Kombat X released for Playstation 5?

TOOL USAGE:

1. retrieve_game
   Arguments: {"query": "Pokémon Gold and Silver release date"}
   Result: [{"Name": "Pokémon Gold and Silver", "YearOfRelease": 1999, "Description": "Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.", "Platform": "Game Boy Color", "Publisher": "Nintendo", "Genre": "Role-playing", "similarity": 0.866}, {"Description": "Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.", "YearOfRelease

### (Optional) Advanced

In [13]:
#### Long-Term Memory + Tool-Based State Machine

This advanced version:
1. **Persists useful answers** in a ChromaDB long-term memory store (including web-search learnings)
2. **Runs a fixed workflow** where each tool is a predefined state-machine node:
   `load_memory → retrieve_game → evaluate_retrieval → (web_search if needed) → generate_answer → save_memory`

In [ ]:
from typing import Optional, TypedDict, Union

from lib.memory import LongTermMemory, MemoryFragment
from lib.messages import SystemMessage, UserMessage
from lib.state_machine import EntryPoint, Resource, StateMachine, Step, Termination
from lib.vector_db import VectorStoreManager


class UdaPlayWorkflowState(TypedDict):
    user_query: str
    session_id: str
    owner: str
    memory_context: List[str]
    retrieved_docs: List[Dict[str, Any]]
    evaluation: Dict[str, Any]
    web_results: Optional[Dict[str, Any]]
    messages: List[BaseMessage]
    answer: str
    workflow_trace: List[str]


memory_manager = VectorStoreManager(
    openai_api_key=OPENAI_API_KEY,
    persist_path=CHROMA_PATH,
    openai_api_base=OPENAI_BASE_URL,
)
long_term_memory = LongTermMemory(memory_manager)

answer_llm = LLM(
    model="gpt-4o-mini",
    temperature=0.2,
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
)


class UdaPlayWorkflowAgent:
    """Stateful agent where each tool is a predefined workflow node."""

    def __init__(self, owner: str = "udaplay-user"):
        self.owner = owner
        self.workflow = self._build_workflow()
        self.resource = Resource(
            vars={
                "ltm": long_term_memory,
                "answer_llm": answer_llm,
            }
        )

    def _load_memory(self, state: UdaPlayWorkflowState, resource: Resource) -> Dict[str, Any]:
        search = resource.vars["ltm"].search(
            query_text=state["user_query"],
            owner=state["owner"],
            namespace="udaplay",
            limit=3,
        )
        memory_context = [fragment.content for fragment in search.fragments]
        trace = state.get("workflow_trace", []) + [
            f"load_memory: found {len(memory_context)} relevant memories"
        ]
        return {"memory_context": memory_context, "workflow_trace": trace}

    def _retrieve_step(self, state: UdaPlayWorkflowState, resource: Resource) -> Dict[str, Any]:
        docs = retrieve_game(query=state["user_query"])
        trace = state["workflow_trace"] + [f"retrieve_game: returned {len(docs)} documents"]
        return {"retrieved_docs": docs, "workflow_trace": trace}

    def _evaluate_step(self, state: UdaPlayWorkflowState, resource: Resource) -> Dict[str, Any]:
        evaluation = evaluate_retrieval(
            question=state["user_query"],
            retrieved_docs=state["retrieved_docs"],
        )
        trace = state["workflow_trace"] + [
            f"evaluate_retrieval: useful={evaluation.get('useful')}"
        ]
        return {"evaluation": evaluation, "workflow_trace": trace}

    def _web_search_step(self, state: UdaPlayWorkflowState, resource: Resource) -> Dict[str, Any]:
        web_results = game_web_search(question=state["user_query"])
        trace = state["workflow_trace"] + ["game_web_search: fetched web results"]
        return {"web_results": web_results, "workflow_trace": trace}

    def _generate_answer(self, state: UdaPlayWorkflowState, resource: Resource) -> Dict[str, Any]:
        llm = resource.vars["answer_llm"]
        memory_block = "\n".join(f"- {item}" for item in state["memory_context"]) or "None"
        internal_block = json.dumps(state["retrieved_docs"], indent=2)
        evaluation_block = json.dumps(state["evaluation"], indent=2)
        web_block = json.dumps(state.get("web_results") or {}, indent=2)

        messages = [
            SystemMessage(
                content=(
                    "You are UdaPlay. Write a concise, well-cited final answer using the best source. "
                    "Prefer internal database results when evaluation says they are useful."
                )
            ),
            UserMessage(
                content=(
                    f"Question: {state['user_query']}\n\n"
                    f"Long-term memory:\n{memory_block}\n\n"
                    f"Retrieved documents:\n{internal_block}\n\n"
                    f"Retrieval evaluation:\n{evaluation_block}\n\n"
                    f"Web search results:\n{web_block}\n\n"
                    "Provide the final answer with citations."
                )
            ),
        ]

        response = llm.invoke(messages)
        trace = state["workflow_trace"] + ["generate_answer: produced final response"]
        return {
            "messages": messages + [response],
            "answer": response.content or "",
            "workflow_trace": trace,
        }

    def _save_memory(self, state: UdaPlayWorkflowState, resource: Resource) -> Dict[str, Any]:
        source = "web" if state.get("web_results") else "internal"
        fragment = MemoryFragment(
            content=f"Q: {state['user_query']}\nA: {state['answer']}",
            owner=state["owner"],
            namespace="udaplay",
        )
        resource.vars["ltm"].register(fragment, metadata={"source": source, "session_id": state["session_id"]})
        trace = state["workflow_trace"] + ["save_memory: stored answer in long-term memory"]
        return {"workflow_trace": trace}

    def _build_workflow(self) -> StateMachine[UdaPlayWorkflowState]:
        machine = StateMachine[UdaPlayWorkflowState](UdaPlayWorkflowState)

        entry = EntryPoint[UdaPlayWorkflowState]()
        load_memory = Step[UdaPlayWorkflowState]("load_memory", self._load_memory)
        retrieve = Step[UdaPlayWorkflowState]("retrieve_game", self._retrieve_step)
        evaluate = Step[UdaPlayWorkflowState]("evaluate_retrieval", self._evaluate_step)
        web_search = Step[UdaPlayWorkflowState]("game_web_search", self._web_search_step)
        generate = Step[UdaPlayWorkflowState]("generate_answer", self._generate_answer)
        save_memory = Step[UdaPlayWorkflowState]("save_memory", self._save_memory)
        termination = Termination[UdaPlayWorkflowState]()

        machine.add_steps([
            entry, load_memory, retrieve, evaluate, web_search, generate, save_memory, termination
        ])

        machine.connect(entry, load_memory)
        machine.connect(load_memory, retrieve)
        machine.connect(retrieve, evaluate)

        def route_after_evaluation(
            state: UdaPlayWorkflowState,
        ) -> Union[Step[UdaPlayWorkflowState], str]:
            if state["evaluation"].get("useful"):
                return generate
            return web_search

        machine.connect(evaluate, [generate, web_search], route_after_evaluation)
        machine.connect(web_search, generate)
        machine.connect(generate, save_memory)
        machine.connect(save_memory, termination)

        return machine

    def invoke(self, query: str, session_id: str = "udaplay-advanced") -> Run:
        initial_state: UdaPlayWorkflowState = {
            "user_query": query,
            "session_id": session_id,
            "owner": self.owner,
            "memory_context": [],
            "retrieved_docs": [],
            "evaluation": {},
            "web_results": None,
            "messages": [],
            "answer": "",
            "workflow_trace": [],
        }
        return self.workflow.run(initial_state, self.resource)


workflow_agent = UdaPlayWorkflowAgent(owner="udaplay-user")

In [ ]:
def print_workflow_report(query: str, run: Run) -> None:
    final_state = run.get_final_state() or {}

    print("=" * 80)
    print(f"QUERY: {query}")
    print("=" * 80)

    print("\nWORKFLOW TRACE:")
    for step in final_state.get("workflow_trace", []):
        print(f"- {step}")

    print("\nLONG-TERM MEMORY USED:")
    memory_context = final_state.get("memory_context", [])
    if memory_context:
        for item in memory_context:
            print(f"- {item[:200]}")
    else:
        print("- None")

    print("\nFINAL ANSWER:")
    print(final_state.get("answer", "No answer generated."))

    print("\nRUN METADATA:")
    print(json.dumps(run.metadata, indent=2))
    print("=" * 80)

In [ ]:
# First query: likely needs web search and will be saved to long-term memory
advanced_query_1 = "Was Mortal Kombat X released for Playstation 5?"
advanced_run_1 = workflow_agent.invoke(advanced_query_1, session_id="udaplay-advanced")
print_workflow_report(advanced_query_1, advanced_run_1)

In [ ]:
# Follow-up query: should recall the saved long-term memory
advanced_query_2 = "What did we learn earlier about Mortal Kombat X on Playstation 5?"
advanced_run_2 = workflow_agent.invoke(advanced_query_2, session_id="udaplay-advanced")
print_workflow_report(advanced_query_2, advanced_run_2)